In [0]:
import fastf1
fastf1.Cache.enable_cache('../cache')  # creates a ./cache directory
import pandas as pd

from fastf1 import get_session
from tqdm import tqdm
import numpy as np

In [0]:
session = get_session(2023, 'Silverstone', 'R')  # Year, GP name, session type (R=Race, Q=Quali)
session.load()  # downloads data and parses everything

In [0]:
laps = session.laps  # lap-level data (driver, time, stint, tyre, pit stop)
results = session.results  # final positions, fastest lap, team
weather = session.weather_data  # temp, humidity, wind
track_stats = session.track_status

In [0]:
pd.set_option('display.max_columns', None)

In [0]:
laps.loc[laps["TrackStatus"] != "1"].head(1000)

In [0]:
results.head()

In [0]:
weather.head()

In [0]:
track_stats.head(150)

In [0]:
events = fastf1.get_event_schedule(2018)
events.head()

In [0]:
events = fastf1.get_event_schedule(2020)
events = events.loc[events["EventFormat"] == "conventional"]["Location"]
events = list(set(events.tolist()))
print(events)

In [0]:
years = [2018, 2019, 2020, 2021, 2022, 2023]
event_data = {}
for year in years:
    e = fastf1.get_event_schedule(year)
    e = e.loc[e["EventFormat"] != "testing"]["EventName"]
    races = e.tolist()
    event_data[year] = races

In [0]:
print(event_data)
for year, events in event_data.items():
    print(f"\t{year} - {len(events)} - {events}")

In [ ]:
master_rows = []

for year in tqdm(event_data, desc="Years"):
    for race in tqdm(event_data[year], desc=f"Processing {year}", leave=False):

        try:
            # -------------------------
            # LOAD SESSION (RACE)
            # -------------------------
            session = fastf1.get_session(year, race, 'R')
            session.load()

            laps = session.laps.copy()
            weather = session.weather_data.copy()
            drivers = session.drivers

            # -------------------------
            # CLEAN WEATHER COLUMNS
            # -------------------------
            weather = weather.rename(columns={
                'AirTemp': 'AirTemp_C',
                'TrackTemp': 'TrackTemp_C',
                'Humidity': 'Humidity_pct',
                'WindSpeed': 'WindSpeed_kmh',
                'Rainfall': 'Rainfall_mm'
            })

            # Sort for merge_asof()
            laps_sorted = laps.sort_values("Time")
            weather_sorted = weather.sort_values("Time")

            # -------------------------
            # MERGE WEATHER INTO LAPS
            # -------------------------
            laps_merged = pd.merge_asof(
                laps_sorted,
                weather_sorted,
                on="Time",
                direction="nearest"
            )

            # -------------------------
            # ENSURE TYRE-RELATED FIELDS EXIST
            # -------------------------
            tyre_fields = ['Compound', 'Stint', 'FreshTyre', 'TyreLife']
            for col in tyre_fields:
                if col not in laps_merged.columns:
                    laps_merged[col] = None

            # -------------------------
            # ADD SESSION METADATA
            # -------------------------
            laps_merged['Year'] = year
            laps_merged['Race'] = race
            laps_merged['Circuit'] = session.event['OfficialEventName']
            laps_merged['Location'] = session.event['Location']
            laps_merged['Country'] = session.event['Country']
            laps_merged['SessionDate'] = session.event['EventDate']

            # -------------------------
            # ADD DRIVER METADATA
            # -------------------------
            driver_info = {}
            for drv in drivers:
                info = session.get_driver(drv)
                driver_info[info['Abbreviation']] = {
                    'DriverNumber': info['DriverNumber'],
                    'BroadcastName': info['BroadcastName'],
                    'TeamColor': info['TeamColor'],
                    'TeamName': info['TeamName']
                }

            laps_merged['DriverNumber'] = laps_merged['Driver'].map(
                lambda x: driver_info.get(x, {}).get('DriverNumber')
            )
            laps_merged['TeamName'] = laps_merged['Driver'].map(
                lambda x: driver_info.get(x, {}).get('TeamName')
            )
            laps_merged['TeamColor'] = laps_merged['Driver'].map(
                lambda x: driver_info.get(x, {}).get('TeamColor')
            )
            laps_merged['BroadcastName'] = laps_merged['Driver'].map(
                lambda x: driver_info.get(x, {}).get('BroadcastName')
            )

            # -------------------------
            # TRACK STATUS FLAGS
            # -------------------------
            # 1 = track clear, 2 = yellow, 4 = SC, 5 = VSC, 7 = red flag
            laps_merged['TrackStatus'] = laps_merged['TrackStatus'].fillna("1")

            # -------------------------
            # SECTOR TIMES (important for degradation analysis)
            # -------------------------
            laps_merged['Sector1Time'] = laps_merged['Sector1Time'].fillna(pd.Timedelta(seconds=0))
            laps_merged['Sector2Time'] = laps_merged['Sector2Time'].fillna(pd.Timedelta(seconds=0))
            laps_merged['Sector3Time'] = laps_merged['Sector3Time'].fillna(pd.Timedelta(seconds=0))

            # -------------------------
            # GAP TO CAR AHEAD
            # -------------------------
            # Computed from `Time` (FastF1's cumulative session clock when a
            # lap was completed), not `LapTime` (that lap's own duration).
            # Diffing LapTime only compares how fast two laps were relative
            # to each other; it says nothing about how far apart the cars
            # actually are on track. Diffing Time, sorted by running
            # Position within each lap, gives the real on-track gap.
            laps_merged['Position'] = pd.to_numeric(laps_merged['Position'], errors='coerce')

            _by_pos = laps_merged.sort_values(['LapNumber', 'Position'])
            gap_to_ahead = _by_pos.groupby('LapNumber')['Time'].diff().dt.total_seconds()
            laps_merged['GapToAhead'] = gap_to_ahead.reindex(laps_merged.index)
            laps_merged.loc[laps_merged['Position'].isna(), 'GapToAhead'] = np.nan

            # -------------------------
            # DELTA TO LEADER
            # -------------------------
            # Same fix as above: gap to the race leader (P1) on track,
            # using cumulative Time rather than comparing lap durations.
            leader_time_by_lap = (
                laps_merged.loc[laps_merged['Position'] == 1]
                .drop_duplicates('LapNumber')
                .set_index('LapNumber')['Time']
            )
            laps_merged['DeltaToLeader'] = (
                laps_merged['Time'] - laps_merged['LapNumber'].map(leader_time_by_lap)
            ).dt.total_seconds()
            laps_merged.loc[laps_merged['Position'].isna(), 'DeltaToLeader'] = np.nan

            # -------------------------
            # DELTA TO FIELD AVERAGE
            # -------------------------
            avg_laptimes = laps_merged.groupby("LapNumber")["LapTime"].transform("mean")
            laps_merged["DeltaToAverage"] = (laps_merged["LapTime"] - avg_laptimes)

            # -------------------------
            # PIT STOP PROCESSING (CORRECTED VERSION)
            # -------------------------

            # 1. Ensure PitInTime & PitOutTime are Timedelta
            laps_merged["PitInTime"] = pd.to_timedelta(laps_merged["PitInTime"], errors="coerce").fillna(pd.Timedelta(0))
            laps_merged["PitOutTime"] = pd.to_timedelta(laps_merged["PitOutTime"], errors="coerce").fillna(pd.Timedelta(0))

            # 2. A pit occurs if PitOutTime > PitInTime (FastF1 uses 0 → 0 for non-pit laps)
            laps_merged["HasPit"] = laps_merged["PitOutTime"] > laps_merged["PitInTime"]

            # 3. Compute pit duration ONLY for detected pit laps
            laps_merged["PitDuration"] = np.where(
                laps_merged["HasPit"],
                (laps_merged["PitOutTime"] - laps_merged["PitInTime"]).dt.total_seconds(),
                0
            )

            # Optional: Clean negative/zero durations (bad data in old races)
            laps_merged.loc[laps_merged["PitDuration"] < 0, "PitDuration"] = 0

            # -------------------------
            # RACE LAP NUMBER
            # -------------------------
            laps_merged['RaceLap'] = laps_merged['LapNumber']

            # -------------------------
            # QUALIFYING RESULTS
            # -------------------------
            try:
                quali = fastf1.get_session(year, race, 'Q')
                quali.load()

                quali_results = quali.results[['DriverNumber', 'Position', 'Q1', 'Q2', 'Q3']]
                quali_results = quali_results.rename(columns={
                    'Position': 'QualiPosition',
                    'Q1': 'Q1Time',
                    'Q2': 'Q2Time',
                    'Q3': 'Q3Time'
                })

                laps_merged = laps_merged.merge(
                    quali_results,
                    on='DriverNumber',
                    how='left'
                )

            except Exception as e:
                print(f"⚠️ Qualifying not available for {race} {year}: {e}")
                laps_merged[['QualiPosition', 'Q1Time', 'Q2Time', 'Q3Time']] = None

            # -------------------------
            # FINAL RACE RESULTS
            # -------------------------
            results = session.results[['DriverNumber', 'Position', 'Status', 'Points', 'Time']]
            results = results.rename(columns={
                'Position': 'FinalPosition',
                'Time': 'FinalRaceTime'
            })

            laps_merged = laps_merged.merge(
                results,
                on='DriverNumber',
                how='left'
            )

            # -------------------------
            # APPEND TO MASTER
            # -------------------------
            master_rows.append(laps_merged)
            print("-"*100)
            print(f"Success for {year} {race}!!!!!")
            print("-"*100)

        except Exception as e:
            print(f"⚠️ FAILED FOR {race} {year} → {e}")
            continue


# -------------------------
# CONCATENATE & SAVE
# -------------------------
master_df = pd.concat(master_rows, ignore_index=True)
master_df.to_parquet("../data/f1_master_lap_dataset.parquet", index=False)

print("🎉 COMPLETE DATASET SAVED → ../data/f1_master_lap_dataset.parquet")
print(f"Total rows: {len(master_df)}")